In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pyodbc
import sqlite3
from app.GetWorkFlow.search_requests import (
    agent_permissions,
    absent_agents,
    agent_skillsets,
    agent_skills_config,
    agent_skills_cat,
)

from Elastic.elastic_settings import make_elastic_query

from app.db import DB
from app.GetWorkFlow.routing import routing
from typing import Optional, Tuple 


Current project environment: UAT
Current SP environment: PROD
https://operationcentre.ms.bell.ca
845e3004-c0f2-41c7-845f-2dd89481c842


In [2]:
from app.GetWorkFlow.service import RoutingService
from app.models import ElasticRequests

### SQLite

In [3]:
# db = '../database/bbm_stm_v1.db'

In [4]:
# con = sqlite3.connect(db)
# cur = con.cursor()

### ODBC

In [5]:
import pyodbc

server = 'tcp:CWYPWD-4008.bell.corp.bce.ca'
database = 'bbm_stm'
username = 'FID_STM_ACCESS'
password = 'H7t45#des'

con = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};SERVER='
        +server+';DATABASE='+database+';UID='+username+';PWD='+password
    )
# cur = conn.cursor()
# sql = ("SELECT " 
#             "T1.skill_id "
#         f"FROM {DB.skill_agent_priority()} AS T1 ")
# cur.execute(sql).fetchall()

### Test RoutingService function

In [10]:
e_requests_uat = ElasticRequests('bbm_aiml_stm_uat_v3')

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\elasticsearch\connection\http_urllib3.py:209: UserWarning: Connecting to https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
c:\Users\ez99152\Documents\stm\stm-api\app\GetWorkFlow\search_requests_utils.py:116: ElasticsearchWarning: The client is unable to verify that the server is Elasticsearch due security privileges on the server side
  r = es_connection.search(body=make_elastic_query(date), index=index_name, size=size, request_timeout=30)
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\conne

In [11]:
e_requests_uat.elastic_requests

{'bsd_00006040fb3c00000000001af308': {'source.customer': {'name': 'Primus Telecommunications Canada Inc.',
   'id': '1002670394'},
  'source.customerSupportModel': 'Standard',
  'accessPolicyTag': 'DIVISION_WHOLESALE',
  'workOrder.status': 'standardOrder',
  'source.orderDate': '2022-04-29T16:03:39.062477269Z',
  'source.requestType': 'Add',
  'source.goldenCustomer': '1002670394',
  'source.focTarget': 0,
  'source.requestSource': 'SSC',
  'workOrder.controlDesk': 'wholesale',
  'source.customerMarketSegment': 'Quebec Enterprise',
  'lastUpdated': '2022-05-09 15:10:24.04100+0000',
  'source.status': 'inProgress',
  'source.serviceRegion': 'QC',
  'source.requestedStartDate': '2022-04-29T16:03:39.062477269Z',
  'source.preferredLanguage': 'en',
  'workOrder.followUpDate': '2022-05-04T04:00:00Z',
  'source.businessUnit': 'Wholesale',
  'workOrder.tags': 'SMARTPATH_DISTRIBUTION_MODE_FILTER',
  'request_id': 'bsd_00006040fb3c00000000001af308',
  'workOrder.assignee': {'name': 'Adrian Adm

In [7]:
e_requests = ElasticRequests('bbm_aiml_stm_prod_v2')

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [9]:
e_requests.elastic_requests

{'bsd_REQ-545671': {'source.customer': {'name': 'The Corporation of the City Of Woodstock',
   'id': '1-P2I-5672'},
  'source.customerSupportModel': 'Standard',
  'accessPolicyTag': 'DIVISION_RETAIL',
  'workOrder.status': 'new',
  'source.orderDate': '2022-05-11T19:04:18Z',
  'source.requestType': 'disconnect',
  'source.goldenCustomer': '1004699830',
  'source.focTarget': 2,
  'source.requestSource': 'ssc',
  'workOrder.controlDesk': 'businessCare',
  'source.customerMarketSegment': 'Ontario SMB',
  'lastUpdated': '2022-05-11 19:04:28.88700+0000',
  'source.status': 'acknowledged',
  'source.serviceRegion': 'ON',
  'source.requestedStartDate': '2022-05-11T19:04:18Z',
  'source.preferredLanguage': 'en',
  'source.businessUnit': 'Retail',
  'workOrder.tags': ['CREATED_FROM_SSC SMARTPATH_DISTRIBUTION_MODE_FILTER'],
  'request_id': 'bsd_REQ-545671',
  'source.product': 'businessLines'},
 'bsd_REQ-545670': {'source.customerSupportModel': 'Standard',
  'accessPolicyTag': 'DIVISION_WIRELESS

In [16]:
routing_service = RoutingService(con, 'luc.clement', 'wireless', DB.agent())

In [19]:
routing_service.run(e_requests)

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


BusinessUnit in ('Wireless'):  (0, 23)
accessPolicyTag filter:  (0, 23)
workOrder_tags filter:  (0, 23)
status + followUpDate filter:  (0, 23)
no assignee filter:  (0, 24)
IS assignee filter:  (0, 23)
assignee in all_list filter:  (0, 24)
final df shape:  (0, 24)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (0, 24)
customerSupportModel filters: (0, 24)
controlDesk filters: (0, 24)
preferredlanguage filters: (0, 24)
0 foc filters: (0, 24)
2 foc filters: (0, 24)
4 foc filters: (0, 24)
5 foc filters: (0, 24)
10 foc filters: (0, 24)
15 foc filters: (0, 24)
30 foc filters: (0, 24)
75 foc filters: (0, 24)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (0, 24)
customerSupportModel filters: (0, 24)
controlDesk filters: (0, 24)
preferredlanguage filters: (0, 24)
0 foc filters: (0, 24)
2 foc filters: (0, 24)
4 foc filters: (0, 24)
5 foc filters: (0, 24)
10 foc filters: (0, 24)
15 foc filters: (0, 24)
30 foc filters: (0, 24)


('NO_WORK', None, None)

### Decomposing RoutingService.run() functions 

In [12]:
agent_id = 'luc.clement' # 'bccs.n435345' 466802

In [13]:
# Find the agent skillsets 
skillsets = agent_skillsets(con, agent_id)
skillsets

['6166e891a6a35b0001dd31f1', '6166e8d7a6a35b0001dd31fa']

In [14]:
# Find the permissions of the agent 
permissions = agent_permissions(con, agent_id)
permissions


['OC-SmartPath-NetNewCreator', 'OC-SmartPath-Wireless']

In [15]:
configuration = agent_skills_config(con, skillsets)
configuration

{'6166e891a6a35b0001dd31f1': {'controlDesk': ['hscOttawa'],
  'goldenCustomer': [''],
  'customerMarketSegment': [''],
  'preferredlanguage': ['en'],
  'requestSource': ['email', 'smartpath'],
  'product': [''],
  'serviceRegion': [''],
  'customerSupportModel': ['Non-Standard', 'Standard'],
  'requestType': ['']},
 '6166e8d7a6a35b0001dd31fa': {'controlDesk': ['hscOttawa'],
  'goldenCustomer': [''],
  'customerMarketSegment': [''],
  'preferredlanguage': ['fr'],
  'requestSource': ['email', 'smartpath'],
  'product': [''],
  'serviceRegion': [''],
  'customerSupportModel': ['Non-Standard', 'Standard'],
  'requestType': ['']}}

In [16]:
# Find the splitting on subcategories of skillsets 
split_config = agent_skills_cat(configuration)
split_config

{'6166e891a6a35b0001dd31f1_0': {'skill_id': '6166e891a6a35b0001dd31f1',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Non-Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['en']},
 '6166e891a6a35b0001dd31f1_1': {'skill_id': '6166e891a6a35b0001dd31f1',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['en']},
 '6166e8d7a6a35b0001dd31fa_0': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Non-Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']},
 '6166e8d7a6a35b0001dd31fa_1': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']}}

### Decomposing search_all_requests() functions

In [17]:
from app.GetWorkFlow.search_requests_utils import *
from app.GetWorkFlow.search_requests import agent_permissions
from app.Elastic.elastic_settings import make_elastic_query

#### Tests

In [18]:
import sys
sys.path.append('..')
from app.Elastic.elastic_api_index import initialize_elastic_connection

INDEX_NAME = 'bbm_aiml_stm_prod_v2'
es_connection = initialize_elastic_connection() 


In [19]:
r = es_connection.search(body=make_elastic_query("2020-04-26T18:41:01"), index=INDEX_NAME, size=50_000, request_timeout=30)

C:\Users\ez99152\AppData\Local\Temp/ipykernel_14464/1484883275.py:1: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  r = es_connection.search(body=make_elastic_query("2020-04-26T18:41:01"), index=INDEX_NAME, size=50_000, request_timeout=30)
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\AppData\Local\Temp/ipykernel_14464/1484883275.py:1: ElasticsearchWarning: The client is unable to verify that the server is Elasticsearch due security privileges on the server side
  r = es_connectio

In [20]:
len(r['hits']['hits'])

16033

In [98]:
from datetime import datetime
date = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%dT%H:%M:%S')
date


'2022-05-11T17:17:31'

In [21]:
def clean_resp_dict(resp):
    """Transform the response dict which can have nested fields into a flattened
    dict.
    Note: This works for current fields as of 04/22, and might need to be updated
    if more data fields are ingested."""
    
    cleaned_r_dict = {}

    for hit in resp['hits']['hits']:
        single_cleaned_r = {}
        for k, v in hit['fields'].items():
            if k == 'source.goldenCustomer' and 'id' in v[0].keys():
                single_cleaned_r[k] = v[0]['id']
            else:
                if len(v) == 1:
                    single_cleaned_r[k] = v[0]
                if len(v) > 1:
                    single_cleaned_r[k] = [' '.join(v)]
        cleaned_r_dict[single_cleaned_r['request_id']] = single_cleaned_r

    return cleaned_r_dict


In [22]:
def keep_latest_request(d_all, d_single):

    last_updated_single = datetime.strptime(
        d_single['lastUpdated'], '%Y-%m-%d %H:%M:%S.%f+0000')
    last_updated_all = datetime.strptime(
        d_all[d_single['request_id']]['lastUpdated'], '%Y-%m-%d %H:%M:%S.%f+0000')
        
    if last_updated_single > last_updated_all:
        return d_single
    else:
        return d_all[d_single['request_id']]

In [23]:
test_r1 = clean_resp_dict(r)
test_r2 = clean_resp_dict(r)

In [44]:
test_d1 = test_r1["bsd_REQQA-465409"]
test_d1['lastUpdated'] = "2022-04-20 18:36:35.65700+0000"

test_d2 = test_r1["bsd_REQQA-465408"]
test_d2['lastUpdated'] = "2022-04-28 18:36:35.65700+0000"

KeyError: 'bsd_REQQA-465409'

In [ ]:
test_r1["bsd_REQQA-465409"] = test_d1
test_r1["bsd_REQQA-465408"] = test_d2

In [102]:
def update_response_dict(d_all, d_new):
    for k, v in d_new.items():
        if k in d_all.keys():
            d = keep_latest_request(d_all, v)
            d_all[k] = d
        else:
            d_all[k] = v
    return d_all

In [45]:
final_d = update_response_dict(test_r1, test_r2)

NameError: name 'update_response_dict' is not defined

In [46]:
[d for d in final_d.values()]

NameError: name 'final_d' is not defined

#### get_request_from_elastic()

In [24]:
INDEX_NAME = 'bbm_aiml_stm_prod_v2'
elastic_resp_cleaned = get_request_from_elastic(INDEX_NAME, "2020-04-27T10:20:55", size=100000)

C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [25]:
len(elastic_resp_cleaned)

16033

In [26]:
elastic_resp_cleaned

{'bsd_REQ-545671': {'source.customer': {'name': 'The Corporation of the City Of Woodstock',
   'id': '1-P2I-5672'},
  'source.customerSupportModel': 'Standard',
  'accessPolicyTag': 'DIVISION_RETAIL',
  'workOrder.status': 'new',
  'source.orderDate': '2022-05-11T19:04:18Z',
  'source.requestType': 'disconnect',
  'source.goldenCustomer': '1004699830',
  'source.focTarget': 2,
  'source.requestSource': 'ssc',
  'workOrder.controlDesk': 'businessCare',
  'source.customerMarketSegment': 'Ontario SMB',
  'lastUpdated': '2022-05-11 19:04:28.88700+0000',
  'source.status': 'acknowledged',
  'source.serviceRegion': 'ON',
  'source.requestedStartDate': '2022-05-11T19:04:18Z',
  'source.preferredLanguage': 'en',
  'source.businessUnit': 'Retail',
  'workOrder.tags': ['CREATED_FROM_SSC SMARTPATH_DISTRIBUTION_MODE_FILTER'],
  'request_id': 'bsd_REQ-545671',
  'source.product': 'businessLines'},
 'bsd_REQ-545670': {'source.customerSupportModel': 'Standard',
  'accessPolicyTag': 'DIVISION_WIRELESS

#### elastic_response_to_df()

In [27]:
df_resp = pd.DataFrame(elastic_resp_cleaned, columns=query_fields)


In [28]:
df = elastic_response_to_df(elastic_resp_cleaned, query_fields)
df

c:\Users\ez99152\Documents\stm\stm-api\app\GetWorkFlow\search_requests_utils.py:139: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna('nan')
c:\Users\ez99152\Documents\stm\stm-api\app\GetWorkFlow\search_requests_utils.py:140: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)


,request_id,lastUpdated,accessPolicyTag,workOrder_status,workOrder_followUpDate,workOrder_controlDesk,workOrder_group,workOrder_assignee,workOrder_tags,source_orderDate,...,source_customerSupportModel,source_businessUnit,source_status,source_product,source_serviceRegion,source_goldenCustomer,source_customer,source_customerMarketSegment,source_preferredLanguage,source_focTarget
0,bsd_REQ-545671,2022-05-11 19:04:28.887000+00:00,division_retail,new,NaT,businesscare,nan,nan,[CREATED_FROM_SSC SMARTPATH_DISTRIBUTION_MODE_...,2022-05-11 19:04:18+00:00,...,standard,retail,acknowledged,businesslines,on,1004699830,{'name': 'The Corporation of the City Of Woods...,ontario smb,en,2
1,bsd_REQ-545670,2022-05-11 19:03:56.290000+00:00,division_wireless,new,NaT,hsctoronto,nan,nan,[CREATED_FROM_EMAIL_REQUEST AIML_PROCESSED SMA...,2022-05-11 19:03:47+00:00,...,standard,wireless,acknowledged,nan,nan,nan,nan,nan,en,0
2,bsd_REQ-545669,2022-05-11 19:03:23.491000+00:00,division_wireless,new,NaT,hsctoronto,nan,nan,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_SKIP_NOT...,2022-05-11 19:03:15+00:00,...,standard,wireless,acknowledged,nan,nan,1004696554,{'name': 'Kuehne + Nagel'},ontario enterprise,en,0
3,bsd_REQ-545668,2022-05-11 19:03:17.122000+00:00,division_wireless,new,NaT,hscottawa,nan,nan,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_SKIP_NOT...,2022-05-11 19:03:08+00:00,...,standard,wireless,acknowledged,nan,nan,1004696554,{'name': 'Kuehne + Nagel'},ontario enterprise,en,0
4,bsd_REQ-545667,2022-05-11 19:03:08.260000+00:00,division_wireless,new,NaT,hscottawa,nan,nan,[CREATED_FROM_EMAIL_REQUEST SMARTPATH_SKIP_NOT...,2022-05-11 19:03:00+00:00,...,standard,wireless,acknowledged,nan,nan,1004696554,{'name': 'Kuehne + Nagel'},ontario enterprise,en,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16028,bsd_REQ-3902,2022-04-10 18:02:03.287000+00:00,[DIVISION_RETAIL TEST],standardOrder,2021-11-08,businesscare,nan,"{'name': 'Marie-Claude Gauvreau', 'id': 'marie...",smartpath_distribution_mode_filter,2020-10-07 18:52:58+00:00,...,standard,retail,inprogress,megalink,on,1000000004,"{'name': 'Maya DVT Customer', 'id': '1-125DL9'}",ontario enterprise,en,4
16029,bsd_REQ-3182,2022-04-10 18:03:31.951000+00:00,division_retail,awaitingCustomer,2022-05-31,businesscare,nan,"{'name': 'Maryline Hawkins', 'id': 'bccs.30112...",smartpath_distribution_mode_filter,2020-09-29 18:54:20+00:00,...,standard,retail,pending,ethernet,ab,1004694840,"{'name': 'Ensign Drilling Inc.', 'id': '1-701C...",bell west,en,30
16030,bsd_REQ-2003,2022-04-10 18:05:58.001000+00:00,[DIVISION_RETAIL TEST],standardOrder,2021-12-11,businesscare,nan,"{'name': 'Catherine McBurney', 'id': 'catherin...",smartpath_distribution_mode_filter,2020-09-16 12:36:36+00:00,...,standard,retail,inprogress,businesslines,intl,1000000004,"{'name': 'Maya DVT Customer', 'id': '1-125DL9'}",ontario enterprise,en,2
16031,bsd_REQ-1050,2022-04-29 23:30:10.991000+00:00,[DIVISION_RETAIL TEST],redistribute,2022-04-29,businesscare,nan,nan,smartpath_distribution_mode_filter,2020-08-19 16:40:41+00:00,...,nan,retail,inprogress,businesslines,nan,1000000004,"{'name': 'Maya DVT Customer', 'id': '1-125DL9'}",nan,en,0


In [29]:
df.shape

(16031, 23)

In [30]:
df.isnull().sum()

request_id                         0
lastUpdated                        0
accessPolicyTag                    0
workOrder_status                   0
workOrder_followUpDate          6847
workOrder_controlDesk              0
workOrder_group                    0
workOrder_assignee                 0
workOrder_tags                     0
source_orderDate                   0
source_requestedStartDate          0
source_requestType                 0
source_requestSource               0
source_customerSupportModel        0
source_businessUnit                0
source_status                      0
source_product                     0
source_serviceRegion               0
source_goldenCustomer              0
source_customer                    0
source_customerMarketSegment       0
source_preferredLanguage           2
source_focTarget                   0
dtype: int64

### Test Filter static function

In [31]:
# Check 'new' et "update_received" dans workOrder_stats. TODO

In [32]:
permissions = agent_permissions(con, 'luc_clement')
df_common_filter = filter_static(con, df, '', permissions)


BusinessUnit in ('Wireless'):  (16031, 23)
accessPolicyTag filter:  (16031, 23)
workOrder_tags filter:  (16031, 23)
status + followUpDate filter:  (4870, 23)
no assignee filter:  (2589, 24)
IS assignee filter:  (2281, 23)
assignee in all_list filter:  (3, 24)
final df shape:  (2592, 24)


C:\Users\ez99152\Anaconda3\envs\ocdp-tti-ddp\lib\site-packages\pandas\core\ops\array_ops.py:73: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior.  In a future version these will be considered non-comparable.Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  result = libops.scalar_compare(x.ravel(), y, op)
c:\Users\ez99152\Documents\stm\stm-api\app\GetWorkFlow\search_requests_utils.py:263: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  d1['from_absent_agent'] = False


In [33]:
df_common_filter.accessPolicyTag.value_counts()

retail                 957
bbm                    425
retail,test            384
wholesale              247
bbm,rpa                199
wholesale,trunkside    158
wireless               124
retail,goclevel2        87
bbm,goclevel2            7
bbm,test                 2
retail,goclevel1         1
retail,rpa               1
Name: accessPolicyTag, dtype: int64

In [35]:
df_common_filter.shape

(2592, 24)

### Test split_skill_dict_per_skill() function

In [36]:
skillsets = agent_skillsets(con, 'luc.clement')
skillsets

['6166e891a6a35b0001dd31f1', '6166e8d7a6a35b0001dd31fa']

In [37]:
configuration = agent_skills_config(con, skillsets)
split_config = agent_skills_cat(configuration)
split_config

{'6166e891a6a35b0001dd31f1_0': {'skill_id': '6166e891a6a35b0001dd31f1',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Non-Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['en']},
 '6166e891a6a35b0001dd31f1_1': {'skill_id': '6166e891a6a35b0001dd31f1',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['en']},
 '6166e8d7a6a35b0001dd31fa_0': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Non-Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']},
 '6166e8d7a6a35b0001dd31fa_1': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']}}

In [38]:
splited_skill_dicts = split_skill_dict_per_skill(split_config, skillsets)


In [39]:
splited_skill_dicts[1]

{'6166e8d7a6a35b0001dd31fa_0': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Non-Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']},
 '6166e8d7a6a35b0001dd31fa_1': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestSource': ['email', 'smartpath'],
  'customerSupportModel': ['Standard'],
  'controlDesk': ['hscOttawa'],
  'preferredlanguage': ['fr']}}

In [40]:
list(splited_skill_dicts[0].keys())

['6166e891a6a35b0001dd31f1_0', '6166e891a6a35b0001dd31f1_1']

In [41]:
# Verify that in each dict, the skill id match
same_name = []
for d in splited_skill_dicts:
    d_keys = list(d.keys())
    d_keys = [k[:-2] for k in d_keys]
    assert d_keys[0] == d_keys[1]

### Test find_attrs_for_cat

In [42]:
find_attrs_for_category(splited_skill_dicts[0]['6166e891a6a35b0001dd31f1_0'])

['requestSource', 'customerSupportModel', 'controlDesk', 'preferredlanguage']

### Test find_request_for_skill() function

In [43]:
skill_dict = {'6166e891a6a35b0001dd31f1_0': 
                {   'skill_id': '6166e891a6a35b0001dd31f1',
                    'requestSource': ['email', 'smartpath'],
                    'customerSupportModel': ['Non-Standard'],
                    'controlDesk': ['hscOttawa'],
                    'preferredlanguage': ['en']},
                '6166e891a6a35b0001dd31f1_1': 
                {   'skill_id': '6166e891a6a35b0001dd31f1',
                    'requestSource': ['email', 'smartpath'],
                    'customerSupportModel': ['Standard'],
                    'controlDesk': ['hscOttawa'],
                    'preferredlanguage': ['en']}
    }

In [44]:
df_reqs = find_request_for_skill(df_common_filter, skill_dict)

Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (1672, 24)
customerSupportModel filters: (132, 24)
controlDesk filters: (0, 24)
preferredlanguage filters: (0, 24)
0 foc filters: (0, 24)
2 foc filters: (0, 24)
4 foc filters: (0, 24)
5 foc filters: (0, 24)
10 foc filters: (0, 24)
15 foc filters: (0, 24)
30 foc filters: (0, 24)
75 foc filters: (0, 24)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (1672, 24)
customerSupportModel filters: (1540, 24)
controlDesk filters: (5, 24)
preferredlanguage filters: (4, 24)
0 foc filters: (4, 24)
2 foc filters: (0, 24)
4 foc filters: (0, 24)
5 foc filters: (0, 24)
10 foc filters: (0, 24)
15 foc filters: (0, 24)
30 foc filters: (0, 24)
75 foc filters: (0, 24)


In [45]:
df_reqs.shape

(4, 24)

In [46]:
df_reqs.workOrder_controlDesk.value_counts()

hscottawa    4
Name: workOrder_controlDesk, dtype: int64

In [47]:
df_reqs.columns

Index(['request_id', 'lastUpdated', 'accessPolicyTag', 'workOrder_status',
       'workOrder_followUpDate', 'workOrder_controlDesk', 'workOrder_group',
       'workOrder_assignee', 'workOrder_tags', 'source_orderDate',
       'source_requestedStartDate', 'source_requestType',
       'source_requestSource', 'source_customerSupportModel',
       'source_businessUnit', 'source_status', 'source_product',
       'source_serviceRegion', 'source_goldenCustomer', 'source_customer',
       'source_customerMarketSegment', 'source_preferredLanguage',
       'source_focTarget', 'from_absent_agent'],
      dtype='object')

In [48]:
df_reqs.head()

,request_id,lastUpdated,accessPolicyTag,workOrder_status,workOrder_followUpDate,workOrder_controlDesk,workOrder_group,workOrder_assignee,workOrder_tags,source_orderDate,...,source_businessUnit,source_status,source_product,source_serviceRegion,source_goldenCustomer,source_customer,source_customerMarketSegment,source_preferredLanguage,source_focTarget,from_absent_agent
3,bsd_REQ-545668,2022-05-11 19:03:17.122000+00:00,wireless,new,NaT,hscottawa,nan,nan,"fromemailrequest,skipnotification,processed,di...",2022-05-11 19:03:08+00:00,...,wireless,acknowledged,nan,nan,1004696554,{'name': 'Kuehne + Nagel'},ontario enterprise,en,0,False
4,bsd_REQ-545667,2022-05-11 19:03:08.260000+00:00,wireless,new,NaT,hscottawa,nan,nan,"fromemailrequest,skipnotification,processed,di...",2022-05-11 19:03:00+00:00,...,wireless,acknowledged,nan,nan,1004696554,{'name': 'Kuehne + Nagel'},ontario enterprise,en,0,False
21,bsd_REQ-545636,2022-05-11 18:57:39.496000+00:00,wireless,new,NaT,hscottawa,nan,nan,"fromemailrequest,skipnotification,processed,di...",2022-05-11 18:57:34+00:00,...,wireless,acknowledged,nan,nan,nan,nan,nan,en,0,False
33,bsd_REQ-545608,2022-05-11 18:51:20.651000+00:00,wireless,new,NaT,hscottawa,nan,nan,"fromemailrequest,processed,distributionmodefilter",2022-05-11 18:51:12+00:00,...,wireless,acknowledged,nan,nan,1005019441,{'name': 'Nav Canada'},quebec enterprise,en,0,False


In [49]:
df_reqs.request_id.duplicated().sum()

0

### Take the latest request from duplicate request id's only

In [50]:
df_common_filter.accessPolicyTag.value_counts()

retail                 957
bbm                    425
retail,test            384
wholesale              247
bbm,rpa                199
wholesale,trunkside    158
wireless               124
retail,goclevel2        87
bbm,goclevel2            7
bbm,test                 2
retail,goclevel1         1
retail,rpa               1
Name: accessPolicyTag, dtype: int64

In [51]:
df_common_filter.accessPolicyTag.value_counts()

retail                 957
bbm                    425
retail,test            384
wholesale              247
bbm,rpa                199
wholesale,trunkside    158
wireless               124
retail,goclevel2        87
bbm,goclevel2            7
bbm,test                 2
retail,goclevel1         1
retail,rpa               1
Name: accessPolicyTag, dtype: int64

In [52]:
df_common_filter.sort_values('lastUpdated')[['lastUpdated', 'accessPolicyTag', 'request_id']].drop_duplicates('accessPolicyTag', keep='last')

,lastUpdated,accessPolicyTag,request_id
1981,2022-04-10 02:22:26.707000+00:00,"retail,rpa",bsd_REQ-457118
2350,2022-04-10 10:29:37.658000+00:00,"bbm,test",bsd_REQ-223802
244,2022-05-11 15:46:30.372000+00:00,"retail,goclevel1",bsd_REQ-544856
134,2022-05-11 17:55:37.427000+00:00,"wholesale,trunkside",bsd_REQ-545226
41,2022-05-11 18:44:56.637000+00:00,"retail,test",bsd_REQ-545584
174,2022-05-11 18:55:16.746000+00:00,"bbm,goclevel2",bsd_REQ-545084
146,2022-05-11 18:56:52.151000+00:00,"retail,goclevel2",bsd_REQ-545158
15,2022-05-11 18:59:58.577000+00:00,wholesale,bsd_REQ-545649
81,2022-05-11 19:00:11.534000+00:00,bbm,bsd_REQ-545436
1801,2022-05-11 19:01:54.911000+00:00,"bbm,rpa",bsd_REQ-499442


### Elastic Query

In [28]:
import sys
import pandas as pd

sys.path.append('..')

from elasticsearch import Elasticsearch

from Elastic.elastic_settings import (
    query,
    INDEX_NAME,
    query_fields,
)

def initialize_elastic_connection():
    
    es_connection = Elasticsearch('https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200/', 
                                  verify_certs = False,
                                  http_auth=('svc-bbm_aiml-user', 'v^*SRFP%uBM2v2bs9VG'))
    return es_connection

In [29]:
def clean_resp_dict(resp):
    """Transform the response dict which can have nested fields into a flattened
    dict.
    Note: This works for current fields as of 04/22, and might need to be updated
    if more data fields are ingested."""
    
    cleaned_r_list = []

    for hit in resp['hits']['hits']:
        single_cleaned_r = {}
        for k, v in hit['fields'].items():
            if k == 'source.goldenCustomer' and 'id' in v[0].keys():
                single_cleaned_r[k] = v[0]['id']
            else:
                if len(v) == 1:
                    single_cleaned_r[k] = v[0]
                if len(v) > 1:
                    single_cleaned_r[k] = [' '.join(v)]
        cleaned_r_list.append(single_cleaned_r)

    return cleaned_r_list


In [37]:
def get_request_from_elastic(query, index_name):
    
    es_connection = initialize_elastic_connection() 
    r = es_connection.search(body=query, index=index_name, size=1_000_000)
    return clean_resp_dict(r)

### Data Preprocessing

In [85]:
def format_df_for_search(df):
    df.columns = [c.replace('.', '_') for c in query_fields]

    df['source_requestedStartDate'] = pd.to_datetime(df.source_requestedStartDate)
    df['workOrder_followUpDate'] = pd.to_datetime(df.workOrder_followUpDate)
    df['source_orderDate'] = pd.to_datetime(df.source_orderDate)
    df['workOrder_followUpDate'] = df['workOrder_followUpDate'].dt.date

    nan_cols = ['workOrder_controlDesk', 'workOrder_assignee', 'source_requestType', 'workOrder_group ',
            'source_customerSupportModel', 'source_product', 'source_serviceRegion', 'source_status',
            'source_goldenCustomer', 'source_customer', 'source_customerMarketSegment', 
            'source_requestSource', 'source_businessUnit']
    
    # df['workOrder_followUpDate'] = df.workOrder_followUpDate.fillna(pd.Timedelta(seconds=0)) 
    for col in nan_cols:
        df[col] = df[col].fillna('nan')
        df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)

    return df

In [86]:
def elastic_response_to_df(response, query_fields):
    df = pd.DataFrame(response, columns=query_fields)
    return format_df_for_search(df)

In [33]:
def clean_tags(x):
    if isinstance(x, list):
        x = x[0].split(' ')
        x1 =[]
        #remove all underscore
        for s in x:
            stmp = s.split('_')
            if len(stmp) ==2:
                x1.append(stmp[1].lower())
            elif len(stmp)>=3:
                x1.append("".join(stmp[1:]).lower())
            else:
                x1.append(s.lower())
        #print("the x1 is", x1)
        x11 = ','.join(map(str, x1))
        return x11
    else:
        stmp = x.split('_')
        if len(stmp) ==2:
            x= stmp[1].lower()
        elif len(stmp)>=3:
            x = "".join(stmp[1:]).lower()
        else:
            x= x.lower()
        return x

def clean_tags_column(df):
    # this function will replace the old values in the dataframe with correct clened values and in a form of hashable list
    df['accessPolicyTag'] = df.accessPolicyTag.apply(lambda x: clean_tags(x))
    df['workOrder_tags'] = df['workOrder_tags'].apply(lambda x: clean_tags(x))
    return df

def pre_process(permission_list):
    #pre-processing on the permission list
    #example permissions = ['OC-SmartPath-GoC-Level2','OC-SmartPath-Retail' , 'OC-SmartPath-BBM' ]
    #remove the unnecessary words
    x1 =[]
    #remove all underscore
    for s in permission_list:
        stmp = s.split('-')
    
        if len(stmp)>=3:
            x1.append("".join(stmp[2:]).lower())
        else:
            x1.append(s.lower())
    return x1

### Common filter for all cats

In [34]:
def filter_static(con, df, tenant, permissions):
    #call the pre-prcessing of permission
    permis_list = pre_process(permissions)
    #get all ids 
    all_ids = absent_agents(con)

    if tenant.lower() == 'wireless':
        #filter from the df business unit wireless
        df_tmp = df.loc[df['source_businessUnit'] == 'wireless']
    elif len(tenant) > 0 and tenant.lower() != 'wireless':
        # get all requests except for wireless
        df_tmp = df.loc[df['source_businessUnit'] != 'wireless']
    else:
        df_tmp = df.copy()
    # We re-write the access policy tags and workorder tags
    #instead of of a list separated by a space bar, we make it as a string separated with comma
    df_tmp_c = clean_tags_column(df_tmp)

    # 1 - let us filter the new dataframe according to a filter list example permission and access policy tag
    contain_policytag = '|'.join(permis_list)

    df_tmp_c = df_tmp_c[
            (pd.notna(df_tmp_c.accessPolicyTag)) & 
            (df_tmp_c.accessPolicyTag.str.contains(contain_policytag))
        ]

    print("accessPolicyTag filter: ", df_tmp_c.shape)

    #2 - let us filter the new dataframe according to the workorder tags
    filter_list = ['distributionmodefilter']
    contain_work_tag ='|'.join(filter_list)

    df_tmp_c = df_tmp_c[
            (pd.notna(df_tmp_c['workOrder_tags'])) &
            (df_tmp_c['workOrder_tags'].str.contains(contain_work_tag))
        ]

    print("workOrder_tags filter: ", df_tmp_c.shape)
    
    # df_tmp_c at this level will only includde workodered distribution mode filter

    #3- we filter using workorder status and followup update
    today = pd.Timestamp.today() 
    df_tmp_c1 = df_tmp_c[
        (df_tmp_c['workOrder_status'].isin(['new','update_received'])) |
        ( 
            (df_tmp_c['workOrder_status']=='parked') &
            (df_tmp_c['workOrder_followUpDate'] <= today)
        )
    ]
    print("status + followUpDate filter: ", df_tmp_c1.shape)

    # 4- Let us check the assignee get the first dataframe containing only no assignees.
    d1 = df_tmp_c1.loc[(df_tmp_c1['workOrder_assignee']=='nan')]

    print("no assignee filter: ", d1.shape)

    # adding a column that says that this comes from an agent or not
    d1['from_agent'] = False
    d2 = df_tmp_c1.loc[df_tmp_c1['workOrder_assignee']!='nan']

    print("IS assignee filter: ", d2.shape)
    # adding a column that says that this comes from an agent or not
    d2['from_agent'] = True
    #filter over this d2to fnd the id of the agents that are  in all list 
    d2 =d2.loc[d2['workOrder_assignee'].str['id'].isin(all_ids)]
    print("assignee in all_list filter: ", d2.shape)

    df_final_static = pd.concat([d1,d2], ignore_index=True)
    del d1
    del d2 
    print("final df shape: ", df_final_static.shape)
    
    return df_final_static

### Filtering on categories

In [35]:
def split_skill_dict_per_skill(split_config: dict, skillsets: list) -> list:
    """
    `split_config` is a dict containing categories of possibly multiple skills.
    Instead of having 1 single dict for all different agent skills, we create a list of dicts.
    Each dict in the list will only have the categories of 1 skill. 

    This makes looping over each skill easier.
    """
    
    skill_cat_list = []
    for skill_number in skillsets:
        single_skill_dict = {}
        for key, value in split_config.items():
            if skill_number in key:
                single_skill_dict.update({key:value})

        skill_cat_list.append(single_skill_dict)
    return skill_cat_list

In [36]:
def make_cat_values_lower(category: dict) -> dict:

    for k, v in category.items():
        if v and k != 'skill_id':
            category[k] = [s.lower() for s in v]
    return category

In [37]:
def find_attrs_for_category(category: dict) -> list:
    """
    Find category attributes that have values and values are non-empty.
    These attributes will then be used to filter the DataFrame.
    """
    
    category = {k: v for k, v in category.items() if v}
    cat_list = [k for k in category.keys() if k != 'skill_id']
    return cat_list



In [38]:
def find_request_for_skill(df: pd.DataFrame, skill_categories: dict) -> pd.Series:
    """
    1. Initialize a dataframe associated with 1 skill and make a copy of all requests (df).
    2. Loop over each skill category of this 1 skill
        - Get all attributes of this category that exist and are non empty
        - Filter all requests (represented in df) depending on the values of each attribute
            -> We now have a DataFrame filtered on one categories' attributes
        - Loop over each possible foc_target value. For each value, further filter the df
          on the foc target value.
            -> We now have a DataFrame filtered on one categories' attributes + 1 foc target value.
               Let's call this df_foc.
        - Concatenate each df_foc with the initialized empty df_skill.
    -> We have 1 final DataFrame (df_skill) that has all requests for each category

    3. Sort all requests by requestStartDate and keep the oldest one        
    """
    df_reqs = pd.DataFrame(columns=df.columns)

    for category in skill_categories.values():
        print(f"Filtering the following category: {category['skill_id']}")
        df_cat = df.copy()
        category = make_cat_values_lower(category)
        cat_attrs = find_attrs_for_category(category)
        
        for cat_attr, col_name in attr_to_col_mapping.items():
            if cat_attr in cat_attrs:
                df_cat = df_cat[df_cat[col_name].isin(category[cat_attr])]
                print(f"{cat_attr} filters: {df_cat.shape}" )
                
        for foc_target in foc_target_vals:
            df_foc = df_cat[df_cat.source_focTarget == foc_target]
            print(f"{foc_target} foc filters: {df_foc.shape}")
            df_reqs = pd.concat([df_reqs, df_foc], axis=0)
    return df_reqs

In [39]:
# Map attribute names in split_config to its equivalent column name in the DataFrame
attr_to_col_mapping = {
       'requestSource'         : 'source_requestSource',
       'product'               : 'source_product',
       'serviceRegion'         : 'source_serviceRegion',
       'customerSupportModel'  : 'source_customerSupportModel',
       'controlDesk'           : 'workOrder_controlDesk',
       'goldenCustomer'        : 'source_goldenCustomer',
       'customerMarketSegment' : 'source_customerMarketSegment',
       'preferredlanguage'     : 'source_preferredLanguage'
}

# All possible foc target values
foc_target_vals = [0,2,4,5,10,15,30,75]

In [41]:
from typing import Dict, List, Optional, Tuple

def search_request(
        split_config: Dict,
        permissions: List[str],
        # agent_id: str,
        skillsets: list,
        tenant: Optional[str] = ''
    ) -> Dict:
    """
    1. Get a cleaned Elastic response containing all request ids. 
    """

    search_dict = {}

    elastic_resp_cleaned = get_request_from_elastic(query, INDEX_NAME)
    df = elastic_response_to_df(elastic_resp_cleaned, query_fields)

    df_common_filter = filter_static(con, df, tenant, permissions)

    splited_skill_dicts = split_skill_dict_per_skill(split_config, skillsets)

    for skill_dict, skill in zip(splited_skill_dicts, skillsets):
        df_reqs = find_request_for_skill(df_common_filter, skill_dict)
        if df_reqs.shape[0] == 0:
            print(f"no requests found for {skill}")
            continue

        reqs_series = df_reqs.sort_values(by='source_requestedStartDate').iloc[0]

        search_dict.update({
            reqs_series.request_id: {
                "skill_id"             : skill,
                "requestStartDate"     : reqs_series.source_requestedStartDate,
                "requestSource"        : reqs_series.source_requestSource,
                "customerSupportModel" : reqs_series.source_customerSupportModel,
                "requestType"          : reqs_series.source_requestType,
                "focTarget"            : reqs_series.source_focTarget,
                "agent_is_absent"      : reqs_series.from_agent
                }
            }
        )
    return search_dict

In [42]:
search_request(
        split_config,
        permissions,
        # agent_id: str,
        skillsets,
        tenant=''
)

C:\Users\ez99152\Anaconda3\envs\stm\lib\site-packages\elasticsearch\connection\http_urllib3.py:209: UserWarning: Connecting to https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\stm\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


accessPolicyTag filter:  (1235, 22)
workOrder_tags filter:  (1235, 22)
status + followUpDate filter:  (1162, 22)
no assignee filter:  (1162, 22)
IS assignee filter:  (0, 22)
assignee in all_list filter:  (0, 23)
final df shape:  (1162, 23)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (1162, 23)
customerSupportModel filters: (0, 23)
controlDesk filters: (0, 23)
preferredlanguage filters: (0, 23)
0 foc filters: (0, 23)
2 foc filters: (0, 23)
4 foc filters: (0, 23)
5 foc filters: (0, 23)
10 foc filters: (0, 23)
15 foc filters: (0, 23)
30 foc filters: (0, 23)
75 foc filters: (0, 23)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (1162, 23)
customerSupportModel filters: (1162, 23)
controlDesk filters: (205, 23)
preferredlanguage filters: (177, 23)
0 foc filters: (177, 23)
2 foc filters: (0, 23)
4 foc filters: (0, 23)
5 foc filters: (0, 23)
10 foc filters: (0, 23)
15 foc filters: (0, 23)
30 foc filters: (0, 23)
75 foc fi

C:\Users\ez99152\Anaconda3\envs\stm\lib\site-packages\pandas\core\ops\array_ops.py:73: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable. Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  result = libops.scalar_compare(x.ravel(), y, op)



75 foc filters: (0, 23)
Filtering the following category: 6166e8d7a6a35b0001dd31fa
requestSource filters: (1162, 23)
customerSupportModel filters: (1162, 23)
controlDesk filters: (205, 23)
preferredlanguage filters: (28, 23)
0 foc filters: (28, 23)
2 foc filters: (0, 23)
4 foc filters: (0, 23)
5 foc filters: (0, 23)
10 foc filters: (0, 23)
15 foc filters: (0, 23)
30 foc filters: (0, 23)
75 foc filters: (0, 23)


{'bsd_REQQA-465842': {'skill_id': '6166e891a6a35b0001dd31f1',
  'requestStartDate': Timestamp('2022-03-22 15:16:59+0000', tz='UTC'),
  'requestSource': 'smartpath',
  'customerSupportModel': 'standard',
  'requestType': 'changeimei',
  'focTarget': 0,
  'agent_is_absent': False},
 'bsd_REQQA-458824': {'skill_id': '6166e8d7a6a35b0001dd31fa',
  'requestStartDate': Timestamp('2022-04-01 17:21:22+0000', tz='UTC'),
  'requestSource': 'email',
  'customerSupportModel': 'standard',
  'requestType': 'nan',
  'focTarget': 0,
  'agent_is_absent': False}}

In [836]:
search_dict = {}

elastic_resp_cleaned = get_request_from_elastic(query, INDEX_NAME)
df = elastic_response_to_df(elastic_resp_cleaned, query_fields)

df_common_filter = filter_static(con, df, '', permissions)

splited_skill_dicts = split_skill_dict_per_skill(split_config)

for skill_dict, skill in zip(splited_skill_dicts, skillsets):
    df_reqs = find_request_for_skill(df_common_filter, skill_dict)
    if df_reqs.shape[0] == 0:
        print(f"no requests found for {skill}")
        continue

    reqs_series = df_reqs.sort_values(by='source_requestedStartDate').iloc[0]

    search_dict.update({
        reqs_series.request_id: {
            "skill_id"             : skill,
            "requestStartDate"     : reqs_series.source_requestedStartDate,
            "requestSource"        : reqs_series.source_requestSource,
            "customerSupportModel" : reqs_series.source_customerSupportModel,
            "requestType"          : reqs_series.source_requestType,
            "focTarget"            : reqs_series.source_focTarget,
            "agent_is_absent"      : reqs_series.from_agent
            }
        }
    )

C:\Users\ez99152\Anaconda3\envs\stm\lib\site-packages\elasticsearch\connection\http_urllib3.py:209: UserWarning: Connecting to https://paas-mtrl-blmcih-data-03.qc.bell.ca:9200 using SSL with verify_certs=False is insecure.
  warnings.warn(
C:\Users\ez99152\Anaconda3\envs\stm\lib\site-packages\urllib3\connectionpool.py:1043: InsecureRequestWarning: Unverified HTTPS request is being made to host 'paas-mtrl-blmcih-data-03.qc.bell.ca'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (9635, 22)
customerSupportModel filters: (95, 22)
controlDesk filters: (0, 22)
preferredlanguage filters: (0, 22)
0 foc filters: (0, 22)
2 foc filters: (0, 22)
4 foc filters: (0, 22)
5 foc filters: (0, 22)
10 foc filters: (0, 22)
15 foc filters: (0, 22)
30 foc filters: (0, 22)
75 foc filters: (0, 22)
Filtering the following category: 6166e891a6a35b0001dd31f1
requestSource filters: (9635, 22)
customerSupportModel filters: (9539, 22)
controlDesk filters: (229, 22)
preferredlanguage filters: (201, 22)
0 foc filters: (201, 22)
2 foc filters: (0, 22)
4 foc filters: (0, 22)
5 foc filters: (0, 22)
10 foc filters: (0, 22)
15 foc filters: (0, 22)
30 foc filters: (0, 22)
75 foc filters: (0, 22)
Filtering the following category: 6166e8d7a6a35b0001dd31fa
requestSource filters: (9635, 22)
customerSupportModel filters: (95, 22)
controlDesk filters: (0, 22)
preferredlanguage filters: (0, 22)
0 foc filters: (0, 22)
2 foc

In [816]:
search_dict

{}

### Individual function tests

In [ ]:
splited_skill_dicts = split_skill_dict_per_skill(split_config)


In [ ]:
splited_skill_dicts = split_skill_dict_per_skill(split_config)

skill_1 = splited_skill_dicts[0]['6166e891a6a35b0001dd31f1_0']
skill_1 = make_cat_values_lower(skill_1)
skill_2 = splited_skill_dicts[0]['6166e891a6a35b0001dd31f1_1']
skill_2 = make_cat_values_lower(skill_2)

In [ ]:
skill_1

{'skill_id': '6166e891a6a35b0001dd31f1',
 'requestSource': ['email', 'smartpath'],
 'customerSupportModel': ['non-standard'],
 'controlDesk': ['hscottawa'],
 'preferredlanguage': ['en']}

In [ ]:
cat_attrs = find_attrs_for_category(skill_2)
cat_attrs

['requestSource', 'customerSupportModel', 'controlDesk', 'preferredlanguage']

In [ ]:
df_cat = df.copy()


In [ ]:
df_skill = find_request_for_skill(df, splited_skill_dicts[0])

category filters:  (9635, 22)
category filters:  (95, 22)
category filters:  (0, 22)
category filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
category filters:  (9635, 22)
category filters:  (9539, 22)
category filters:  (229, 22)
category filters:  (201, 22)
foc filters:  (201, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)
foc filters:  (0, 22)


In [51]:

a = 1

In [58]:

def test():

    global a
    print(a)
    a = 2
    print(a)


In [62]:
test()

1
2


In [60]:
a

2

In [61]:
a = 1